# D3_05 교차검증 · 튜닝 · 임계값

> **오늘 질문:** 모델 설정과 판정 기준을 어떻게 정해야, 처음 보는 데이터에서도 통할까?

| 단계 | 데이터 | 하는 일 |
|---|---|---|
| 튜닝 | Train (앞 64%) | 교차검증으로 하이퍼파라미터 고르기 |
| 임계값 | Valid (다음 16%) | 고장 판정 기준 정하기 |
| 최종 평가 | Test (마지막 20%) | 한 번만 확인 |

| 표시 | 의미 |
|---|---|
| 📋 제공 | 코드가 들어 있습니다. 실행만 하세요 |
| ✍️ 입력 | 설명을 보고 직접 입력하세요 |
| 💬 해석 | 결과를 보고 답을 적어 보세요 |

## 0. 준비: Data Mart 피처와 세 갈래 분할 📋

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import precision_recall_curve, PrecisionRecallDisplay
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score

df = pd.read_parquet("../../data/factory_table.parquet")
df["temp_diff"] = df["process_temp"] - df["air_temp"]
df["power"] = df["torque"] * df["rpm"]
df["strain"] = df["tool_wear"] * df["torque"]
df["grade_num"] = df["grade"].map({"L": 0, "M": 1, "H": 2})
features = ["air_temp", "process_temp", "rpm", "torque", "tool_wear",
            "vibration", "temp_diff", "power", "strain", "grade_num"]
X, y = df[features], df["failure"]

cut_test = int(len(df) * 0.8)           # 마지막 20%는 Test
cut_valid = int(cut_test * 0.8)          # 그 앞의 20%는 Valid
X_train, y_train = X.iloc[:cut_valid], y.iloc[:cut_valid]
X_valid, y_valid = X.iloc[cut_valid:cut_test], y.iloc[cut_valid:cut_test]
X_test, y_test = X.iloc[cut_test:], y.iloc[cut_test:]
print(f"Train {len(X_train)} / Valid {len(X_valid)} / Test {len(X_test)}")
print(f"고장  Train {y_train.sum()} / Valid {y_valid.sum()} / Test {y_test.sum()}")

## 1. 교차검증 + 그리드 탐색 ✍️

랜덤포레스트의 깊이(`max_depth`)와 잎의 최소 크기(`min_samples_leaf`) 조합 9가지를 5-Fold 교차검증으로 비교합니다.
**`scoring="f1"`** 을 꼭 지정하세요. 기본값(정확도)으로 튜닝하면 1교시의 함정에 빠집니다. (30초 정도 걸립니다)

> 변수 이름: `params`, `grid`

In [ ]:
# 힌트: GridSearchCV(모델, params, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="f1")

조합별 점수를 표로 봅니다. 📋

In [ ]:
cv = pd.DataFrame(grid.cv_results_)[["param_max_depth", "param_min_samples_leaf", "mean_test_score", "std_test_score"]]
cv.sort_values("mean_test_score", ascending=False).round(3)

## 2. 정밀도-재현율 곡선 (Valid) ✍️

튜닝이 끝난 모델로 Valid의 고장 확률을 구하고, 임계값에 따라 정밀도와 재현율이 어떻게 바뀌는지 봅니다.

> 변수 이름: `best`, `proba_valid`

In [ ]:
# 힌트: 모델.predict_proba(X)[:, 1] 은 '고장일 확률'

## 3. 현장 기준으로 임계값 정하기 📋

현장 조건: **"재현율 80% 이상을 지키면서 정밀도가 가장 높은 임계값"**

In [ ]:
prec, rec, thr = precision_recall_curve(y_valid, proba_valid)
table = pd.DataFrame({"임계값": thr, "정밀도": prec[:-1], "재현율": rec[:-1]})
ok = table[table["재현율"] >= 0.8]
best_thr = ok.sort_values("정밀도").iloc[-1]["임계값"]
print(f"선택한 임계값: {best_thr:.2f}")
ok.sort_values("정밀도", ascending=False).head(5).round(3)

## 4. Test에서 한 번만 평가 ✍️

기본 임계값(0.5)과 Valid에서 고른 임계값을 Test에서 비교합니다.

> 변수 이름: `proba_test`, `pred_05`, `pred_thr`

In [ ]:
# 힌트: (확률 >= 임계값).astype(int)

## 5. 놓친 고장은 어떤 고장인가 📋

In [ ]:
test = df.iloc[cut_test:].copy()
test["proba"] = proba_test
missed = test[(test["failure"] == 1) & (test["proba"] < best_thr)]
print(f"놓친 고장 {len(missed)}건의 유형")
missed[["twf", "hdf", "pwf", "osf", "rnf"]].sum()

## 정리 💬

**Q1.** Valid에서 고른 임계값은 Test에서도 재현율을 올렸나요? 놓친 고장의 유형을 보고 이유를 설명해 보세요.

> 답:


**Q2.** 그렇다면 TWF는 어떻게 대응하는 것이 현실적일까요? 모델과 함께 저장해야 할 것은?

> 답:
